#### Import libraries and functions

In [ ]:
import importlib
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import make_pipeline

import hierarchical_position_classifier as hierarchical_position_classifier_module

importlib.reload(hierarchical_position_classifier_module)

from hierarchical_position_classifier import (
    combine_localization_predictions,
    diagnose_position_split,
    diagnose_position_splits,
    localization_model_summary,
    per_room_hierarchical_summary,
    plot_distance_error_boxplot,
    plot_distance_error_cdf,
    plot_global_position_confusion_matrix,
    plot_localization_error_cdf_by_model,
    plot_position_confusion_by_true_room,
    plot_position_confusion_when_room_correct,
    combine_position_experiment_summaries,
    feature_columns_for_esps,
    plot_position_error_cdf_by_experiment,
    plot_room_specific_confusion_matrix,
    plot_room_specific_error_cdf,
    room_specific_feature_dataframe,
    run_global_position_experiment,
    run_global_position_experiments,
    run_global_position_experiments_by_split,
    run_hierarchical_position_experiment,
    run_hierarchical_position_experiments,
    run_hierarchical_position_experiments_by_split,
    run_room_specific_position_experiment,
    run_room_specific_position_experiments,
    summarize_distance_errors,
    summarize_position_split_diagnostics,
)
from utils import graphs
from utils.csi_preprocessing import process_magnitude_data, summarize_magnitude_processing
from utils.feature_pipeline import (
    bottleneck_esp_counts,
    build_frequency_feature_dataframes,
    feature_dataframe_nan_report,
    feature_window_bottleneck_report,
    summarize_window_bottlenecks,
)
from utils.import_data import get_csv_files, print_user_location_tables, sort_meta_info

# Use the importable implementation so multi-core workers work from notebooks on Windows.
from utils.thesis_csv_processing import (
    lowest_packet_count_files,
    process_csv_files,
    summarize_processing_diagnostics,
)


#### Framework options

In [ ]:
PROJECT_ROOT = Path(r"C:\Users\pedro\OneDrive - Universidade de Coimbra\Ambiente de Trabalho\tese\thesis-project")
DATA_DIR = PROJECT_ROOT / "CSI DATA"
CALIBRATION_MODE = "none"

CSV_PROCESSING_OPTIONS = {
    "max_workers": None,
    "cache_dir": None,
    "use_cache": True,
    "force_reprocess": False,
    "min_rssi_dbm": -95.0,
    "calibration_eps": 1e-12,
}
RETURN_PROCESSING_STATS = False
SHOW_PACKET_COUNTS = False

# GRAPHS
RELOAD_GRAPHS_BEFORE_PLOTS = True
SHOW_RAW_MAGNITUDE_PLOT = False
SHOW_PROCESSED_MAGNITUDE_PLOT = False
SHOW_RANDOM_FOREST_PLOTS = True
SHOW_IQ_PLOTS = False

# PROCESSING
MAGNITUDE_PROCESSING_OPTIONS = {
    "apply_agc_compensation": False,
    "agc_reference": "median",  # median, mean, max, min, or a numeric dB value
    "filter_method": "none",  # none, moving_average, median
    "filter_window": 5,
    "normalization": "none",  # none, center, zscore, minmax, robust
    "epsilon": 1e-8,
}

# FEATURE EXTRACTION
FEATURE_EXTRACTION_OPTIONS = {
    "window_size": 60,
    "overlap_size": 30,
    "calibrate": False,  # preprocessing is handled in the dedicated cell below
    "require_all_esps": False,
}
WINDOW_SIZE = FEATURE_EXTRACTION_OPTIONS["window_size"]
OVERLAP_SIZE = FEATURE_EXTRACTION_OPTIONS["overlap_size"]
REQUIRE_ALL_ESPS = FEATURE_EXTRACTION_OPTIONS["require_all_esps"]

# RF CLASSIFIER TUNING
RANDOM_STATE = 200
TEST_SIZE = 0.30
MIN_STRATIFIED_CLASS_COUNT = 2
MIN_GROUP_SPLIT_COUNT = 2
NON_FEATURE_COLUMNS = {
    "frequency_scenario",
    "scenario",
    "location",
    "user",
    "trial",
    "group_id",
    "window_idx",
    "label",
}
RANDOM_FOREST_PARAMS = {
    "n_estimators": 300,
    "max_features": "sqrt",
    "class_weight": "balanced",
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}

# HIERARCHICAL POSITION CLASSIFIER
HIERARCHICAL_TEST_SIZE = 0.30
HIERARCHICAL_RANDOM_STATE = 42
HIERARCHICAL_ROW_SPACING = 1.0
HIERARCHICAL_COLUMN_SPACING = 1.0
HIERARCHICAL_DISTANCE_UNIT = "meters"
HIERARCHICAL_CONFUSION_DATASET = "Fusion"  # "2.4 GHz", "5 GHz", "Fusion", or "all"
HIERARCHICAL_CONFUSION_NORMALIZE = False
HIERARCHICAL_ROOM_CONFUSION_ANNOTATE = True
HIERARCHICAL_POSITION_CONFUSION_ANNOTATE = False

# IQ GRAPHS
IQ_LOCATION = "Z-0"
IQ_ESPS = "all"  # "all" or a list like ["01", "11"]
IQ_SUBCARRIERS = "all"  # "all", a list of zero-based indices, or text like "0,1,2"
IQ_SAMPLE_STRIDE = 1
IQ_COLUMN_COUNT = 2


In [ ]:
path_to: str = str(PROJECT_ROOT)
path: str = str(DATA_DIR)

# scenario -> location -> user -> esp -> trial -> file_path
FileMap = dict[str, dict[str, dict[str, dict[str, dict[str, Path]]]]]
# scenario -> location -> user -> esp -> trial -> csi ndarray
csi_map = dict[str, dict[str, dict[str, dict[str, dict[str, np.ndarray]]]]]
# scenario -> location -> user -> esp -> trial -> agc gain ndarray
agc_gain_map = dict[str, dict[str, dict[str, dict[str, dict[str, np.ndarray]]]]]

all_data_files = get_csv_files(path)
scenarios_id, locations_id, users_id, esps_id, trials_id = sort_meta_info(path)

print(f"Scenarios present: {', '.join(scenarios_id) or 'none'}")
print_user_location_tables(all_data_files)


In [ ]:
if RETURN_PROCESSING_STATS:
    magnitude_data, agc_gain_data, csv_stats, csv_diagnostics = process_csv_files(
        all_data_files,
        return_stats=True,
        return_diagnostics=True,
        calibration_mode=CALIBRATION_MODE,
        **CSV_PROCESSING_OPTIONS,
    )
    display(csv_stats)
else:
    magnitude_data, agc_gain_data, csv_diagnostics = process_csv_files(
        all_data_files,
        return_diagnostics=True,
        calibration_mode=CALIBRATION_MODE,
        **CSV_PROCESSING_OPTIONS,
    )
    csv_stats = None

if SHOW_PACKET_COUNTS:
    display(summarize_processing_diagnostics(csv_diagnostics))
    display(lowest_packet_count_files(csv_diagnostics, top_n=20))


In [ ]:
if RELOAD_GRAPHS_BEFORE_PLOTS:
    importlib.reload(graphs)

if SHOW_RAW_MAGNITUDE_PLOT:
    graphs.plot_magnitude_analysis_interactive(magnitude_data)


### Magnitude Data Processing

In [ ]:
processed_magnitude_data, magnitude_processing_summary = process_magnitude_data(
    magnitude_data,
    agc_gain_data,
    **MAGNITUDE_PROCESSING_OPTIONS,
)


In [ ]:
magnitude_processing_overview = summarize_magnitude_processing(
    magnitude_processing_summary,
    MAGNITUDE_PROCESSING_OPTIONS,
)
magnitude_processing_overview


In [ ]:
if SHOW_PROCESSED_MAGNITUDE_PLOT:
    if RELOAD_GRAPHS_BEFORE_PLOTS:
        importlib.reload(graphs)

    graphs.plot_magnitude_analysis_interactive(processed_magnitude_data)


### Feature Extraction

In [ ]:
df_24ghz, df_5ghz, df_fusion = build_frequency_feature_dataframes(
    processed_magnitude_data,
    **FEATURE_EXTRACTION_OPTIONS,
)


In [ ]:
feature_dataframes = {
    "2.4 GHz": df_24ghz,
    "5 GHz": df_5ghz,
    "Fusion": df_fusion,
}

feature_dataframe_summary = pd.DataFrame(
    [
        {
            "dataset": dataset_name,
            "rows": dataframe.shape[0],
            "columns": dataframe.shape[1],
            "groups": dataframe["group_id"].nunique() if not dataframe.empty else 0,
            "labels": (
                dict(dataframe["label"].value_counts().sort_index())
                if not dataframe.empty
                else {}
            ),
        }
        for dataset_name, dataframe in feature_dataframes.items()
    ],
)
feature_dataframe_summary

nan_report = feature_dataframe_nan_report(feature_dataframes)
display(nan_report)

if SHOW_PACKET_COUNTS:
    for scenario in ["2.4ghz", "5ghz", "fusion"]:
        report = feature_window_bottleneck_report(
            processed_magnitude_data,
            scenario,
            window_size=WINDOW_SIZE,
            overlap_size=OVERLAP_SIZE,
            require_all_esps=REQUIRE_ALL_ESPS,
        )
        display(summarize_window_bottlenecks(report))
        display(bottleneck_esp_counts(report))
        display(report.sort_values("min_windows").head(20))

In [ ]:
# Position split diagnostics
split_diagnostics = diagnose_position_splits(
    feature_dataframes,
    test_size=0.3,
    random_state=42,
)

display(split_diagnostics)

split_summary = summarize_position_split_diagnostics(split_diagnostics)
display(split_summary)

display(
    split_diagnostics[
        split_diagnostics["test_location_missing_from_train"]
    ].sort_values(["dataset", "room_label", "location"])
)


### Hierarchical Position Classifier

In [ ]:
hier_results, hier_predictions, hier_models = run_hierarchical_position_experiments(
    feature_dataframes,
    test_size=HIERARCHICAL_TEST_SIZE,
    random_state=HIERARCHICAL_RANDOM_STATE,
    row_spacing=HIERARCHICAL_ROW_SPACING,
    column_spacing=HIERARCHICAL_COLUMN_SPACING,
)

display(hier_results)


### Hierarchical Distance Error

In [ ]:
all_hier_predictions = pd.concat(hier_predictions.values(), ignore_index=True)

distance_summary = summarize_distance_errors(all_hier_predictions)
display(distance_summary)

plot_distance_error_boxplot(all_hier_predictions)
plot_distance_error_cdf(all_hier_predictions)


### Hierarchical Per-Room Analysis

In [ ]:
room_summary = per_room_hierarchical_summary(all_hier_predictions)
display(room_summary)

plot_position_confusion_by_true_room(
    all_hier_predictions,
    dataset="Fusion",
)

plot_position_confusion_when_room_correct(
    all_hier_predictions,
    dataset="Fusion",
)


### Global Position Baseline

In [ ]:
global_results, global_predictions, global_models = run_global_position_experiments(
    feature_dataframes,
    test_size=HIERARCHICAL_TEST_SIZE,
    random_state=HIERARCHICAL_RANDOM_STATE,
    row_spacing=HIERARCHICAL_ROW_SPACING,
    column_spacing=HIERARCHICAL_COLUMN_SPACING,
)

display(global_results)

plot_global_position_confusion_matrix(
    global_predictions["Fusion"],
    dataset="Fusion",
)


### Hierarchical vs Global Localization

In [ ]:
combined_predictions = combine_localization_predictions(
    hier_predictions,
    global_predictions,
)

comparison_summary = localization_model_summary(combined_predictions)
display(comparison_summary)

plot_localization_error_cdf_by_model(
    combined_predictions,
    dataset="Fusion",
)


### Hierarchical Confusion Matrices

In [ ]:
selected_hierarchical_datasets = (
    list(hier_predictions)
    if HIERARCHICAL_CONFUSION_DATASET == "all"
    else [HIERARCHICAL_CONFUSION_DATASET]
)

hier_room_confusion_matrices = {}
hier_position_confusion_matrices = {}
crosstab_normalize = "index" if HIERARCHICAL_CONFUSION_NORMALIZE else False
heatmap_format = ".2f" if HIERARCHICAL_CONFUSION_NORMALIZE else "d"


def plot_hierarchical_confusion_matrix(
    matrix: pd.DataFrame,
    *,
    title: str,
    annotate: bool,
) -> None:
    height = max(4, min(18, 0.38 * len(matrix.index) + 2))
    width = max(5, min(22, 0.38 * len(matrix.columns) + 3))
    fig, ax = plt.subplots(figsize=(width, height))
    sns.heatmap(
        matrix,
        annot=annotate,
        fmt=heatmap_format,
        cmap="Blues",
        cbar=True,
        linewidths=0.5,
        linecolor="white",
        ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.tick_params(axis="x", rotation=45)
    ax.tick_params(axis="y", rotation=0)
    fig.tight_layout()
    plt.show()


for dataset_name in selected_hierarchical_datasets:
    pred = hier_predictions[dataset_name]
    room_matrix = pd.crosstab(
        pred["true_room"],
        pred["pred_room"],
        rownames=["true_room"],
        colnames=["pred_room"],
        normalize=crosstab_normalize,
        dropna=False,
    )
    position_matrix = pd.crosstab(
        pred["true_location"],
        pred["pred_location"],
        rownames=["true_location"],
        colnames=["pred_location"],
        normalize=crosstab_normalize,
        dropna=False,
    )

    hier_room_confusion_matrices[dataset_name] = room_matrix
    hier_position_confusion_matrices[dataset_name] = position_matrix

    plot_hierarchical_confusion_matrix(
        room_matrix,
        title=f"{dataset_name} - room confusion matrix",
        annotate=HIERARCHICAL_ROOM_CONFUSION_ANNOTATE,
    )
    plot_hierarchical_confusion_matrix(
        position_matrix,
        title=f"{dataset_name} - position confusion matrix",
        annotate=HIERARCHICAL_POSITION_CONFUSION_ANNOTATE,
    )


### Random Forest Models

Random and group-aware train/test splits for each feature dataframe.

In [ ]:
# Prepares the feature matrix, label vector, and group identifiers for modeling.
def _model_inputs(dataframe: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series, pd.Series]:
    feature_columns = [column for column in dataframe.columns if column not in NON_FEATURE_COLUMNS]
    return dataframe[feature_columns], dataframe["label"].astype(int), dataframe["group_id"]


# Returns a dictionary mapping split names to tuples of (train_indices, test_indices).
def _split_indices(dataframe: pd.DataFrame) -> dict[str, tuple[np.ndarray, np.ndarray]]:
    features, y, groups = _model_inputs(dataframe)
    row_indices = np.arange(len(dataframe))
    stratify_labels = y if y.value_counts().min() >= MIN_STRATIFIED_CLASS_COUNT else None
    random_train_idx, random_test_idx = train_test_split(
        row_indices,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=stratify_labels,
    )

    if groups.nunique() < MIN_GROUP_SPLIT_COUNT:
        msg = "Group split requires at least two unique group_id values."
        raise ValueError(msg)

    group_splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
    )
    group_train_idx, group_test_idx = next(group_splitter.split(features, y, groups))

    return {
        "random": (random_train_idx, random_test_idx),
        "group": (group_train_idx, group_test_idx),
    }


# Fits a Random Forest model on the training data and evaluates it on the test data,
# returning the results and metrics.
def _fit_and_evaluate_random_forest(
    dataset_name: str,
    split_name: str,
    dataframe: pd.DataFrame,
    train_idx: np.ndarray,
    test_idx: np.ndarray,
) -> tuple[dict[str, object], pd.DataFrame, pd.DataFrame, object]:
    features, y, groups = _model_inputs(dataframe)
    model = make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestClassifier(**RANDOM_FOREST_PARAMS),
    )

    features_train = features.iloc[train_idx]
    features_test = features.iloc[test_idx]
    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]
    model.fit(features_train, y_train)
    y_pred = model.predict(features_test)

    labels = sorted(y.unique())
    report = pd.DataFrame(
        classification_report(
            y_test,
            y_pred,
            labels=labels,
            output_dict=True,
            zero_division=0,
        ),
    ).transpose()
    matrix = pd.DataFrame(
        confusion_matrix(y_test, y_pred, labels=labels),
        index=pd.Index(labels, name="actual"),
        columns=pd.Index(labels, name="predicted"),
    )

    result = {
        "dataset": dataset_name,
        "split": split_name,
        "train_rows": len(train_idx),
        "test_rows": len(test_idx),
        "train_groups": groups.iloc[train_idx].nunique(),
        "test_groups": groups.iloc[test_idx].nunique(),
        "train_label_counts": dict(y_train.value_counts().sort_index()),
        "test_label_counts": dict(y_test.value_counts().sort_index()),
        "accuracy": accuracy_score(y_test, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
        "precision_macro": precision_score(
            y_test,
            y_pred,
            average="macro",
            zero_division=0,
        ),
        "recall_macro": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
        "precision_weighted": precision_score(
            y_test,
            y_pred,
            average="weighted",
            zero_division=0,
        ),
        "recall_weighted": recall_score(
            y_test,
            y_pred,
            average="weighted",
            zero_division=0,
        ),
        "f1_weighted": f1_score(y_test, y_pred, average="weighted", zero_division=0),
    }
    return result, report, matrix, model


In [ ]:
rf_result_rows = []
rf_classification_reports = {}
rf_confusion_matrices = {}
rf_models = {}
rf_split_indices = {}

# Iterate over each dataset and split, fit the Random Forest model, and store the results.
for dataset_name, dataframe in feature_dataframes.items():
    if dataframe.empty:
        continue

    for split_name, (train_idx, test_idx) in _split_indices(dataframe).items():
        model_key = (dataset_name, split_name)
        rf_split_indices[model_key] = {
            "train_idx": train_idx,
            "test_idx": test_idx,
        }
        result, report, matrix, model = _fit_and_evaluate_random_forest(
            dataset_name,
            split_name,
            dataframe,
            train_idx,
            test_idx,
        )
        rf_result_rows.append(result)
        rf_classification_reports[model_key] = report
        rf_confusion_matrices[model_key] = matrix
        rf_models[model_key] = model

random_forest_results = pd.DataFrame(rf_result_rows).sort_values(["dataset", "split"])
random_forest_results


In [ ]:
SELECTED_RF_MODEL = ("2.4 GHz", "random")
display(rf_classification_reports[SELECTED_RF_MODEL])


### Random Forest Plots

In [ ]:
if SHOW_RANDOM_FOREST_PLOTS:
    if RELOAD_GRAPHS_BEFORE_PLOTS:
        importlib.reload(graphs)

    random_forest_plot_results = graphs.add_random_forest_report_metrics(
        random_forest_results,
        rf_classification_reports,
    )

    graphs.plot_random_forest_metric_comparison(random_forest_plot_results)
    graphs.plot_random_forest_confusion_matrices(rf_confusion_matrices)
    rf_feature_importances = graphs.plot_random_forest_feature_importances(
        rf_models,
        feature_dataframes,
    )
else:
    random_forest_plot_results = None
    rf_feature_importances = None


### Feature Importance Plots

In [ ]:
if RELOAD_GRAPHS_BEFORE_PLOTS:
    importlib.reload(graphs)

hier_position_importances = graphs.hierarchical_position_feature_importance_frame(
    hier_models,
    feature_dataframes,
)

graphs.plot_hierarchical_position_feature_importances(
    hier_position_importances,
    top_n=20,
)

display(hier_position_importances.head())


### I/Q Plots

In [ ]:
if SHOW_IQ_PLOTS:
    if RELOAD_GRAPHS_BEFORE_PLOTS:
        importlib.reload(graphs)

    if "all_data_files" not in globals():
        all_data_files = get_csv_files(path)

    location_key = graphs.normalize_location_input(IQ_LOCATION)
    if location_key is None:
        raise ValueError(f"Invalid location: {IQ_LOCATION}")

    available_esp_keys = graphs.get_available_esp_keys_for_location_files(
        all_data_files,
        location_key,
    )
    if IQ_ESPS == "all":
        esp_keys = graphs.paired_esp_keys(available_esp_keys)
    else:
        raw_esp_values = (
            re.split(r"[\s,;]+", IQ_ESPS.strip())
            if isinstance(IQ_ESPS, str)
            else IQ_ESPS
        )
        esp_keys = [
            esp_key
            for raw_esp in raw_esp_values
            for esp_key in [graphs.normalize_esp_input(str(raw_esp))]
            if esp_key in available_esp_keys
        ]

    if not esp_keys:
        print(f"No ESPs found for {IQ_LOCATION}")
    else:
        print(f"Plotting {location_key} | ESPs {esp_keys} | subcarriers {IQ_SUBCARRIERS}")
        graphs.plot_selected_iq_constellations(
            all_data_files,
            location_key,
            esp_keys,
            sc_indices=IQ_SUBCARRIERS,
            sample_stride=IQ_SAMPLE_STRIDE,
            column_count=IQ_COLUMN_COUNT,
        )


### Room 2 Position Classifier

A simple Random Forest classifier that predicts the locations inside Room 2 using only the ESPs in that room (ESPs 1 and 11, 2 and 12, and 3 and 13).

In [ ]:
# Room 2 Specific ESPs
allowed_esps_24 = ['esp_01', 'esp_02', 'esp_03']
allowed_esps_5 = ['esp_11', 'esp_12', 'esp_13']
allowed_esps_fusion = allowed_esps_24 + allowed_esps_5

def filter_r2_df(df, allowed_esps):
    metadata_cols = [col for col in df.columns if col in NON_FEATURE_COLUMNS]
    feature_cols = [col for col in df.columns if any(col.startswith(esp) for esp in allowed_esps)]
    df_filtered = df[df['label'] == 2][metadata_cols + feature_cols].copy()
    return df_filtered

# Create the 3 dataframes
df_r2_24ghz = filter_r2_df(df_24ghz, allowed_esps_24)
df_r2_5ghz = filter_r2_df(df_5ghz, allowed_esps_5)
df_r2_fusion = filter_r2_df(df_fusion, allowed_esps_fusion)

feature_dataframes_r2 = {
    "2.4 GHz": df_r2_24ghz,
    "5 GHz": df_r2_5ghz,
    "Fusion": df_r2_fusion,
}

for name, df in feature_dataframes_r2.items():
    print(f"Room 2 {name} dataframe shape: {df.shape}")

# Model inputs helper for Room 2 (target is location)
def _model_inputs_r2(dataframe: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series, pd.Series]:
    feature_columns = [column for column in dataframe.columns if column not in NON_FEATURE_COLUMNS]
    return dataframe[feature_columns], dataframe["location"].astype(str), dataframe["group_id"]

def _split_indices_r2(dataframe: pd.DataFrame) -> dict[str, tuple[np.ndarray, np.ndarray]]:
    features, y, groups = _model_inputs_r2(dataframe)
    row_indices = np.arange(len(dataframe))
    stratify_labels = y if y.value_counts().min() >= MIN_STRATIFIED_CLASS_COUNT else None
    random_train_idx, random_test_idx = train_test_split(
        row_indices,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=stratify_labels,
    )

    if groups.nunique() < MIN_GROUP_SPLIT_COUNT:
        msg = "Group split requires at least two unique group_id values."
        raise ValueError(msg)

    group_splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
    )
    group_train_idx, group_test_idx = next(group_splitter.split(features, y, groups))

    return {
        "random": (random_train_idx, random_test_idx),
        "group": (group_train_idx, group_test_idx),
    }

def _fit_and_evaluate_random_forest_r2(
    dataset_name: str,
    split_name: str,
    dataframe: pd.DataFrame,
    train_idx: np.ndarray,
    test_idx: np.ndarray,
) -> tuple[dict[str, object], pd.DataFrame, pd.DataFrame, object]:
    features, y, groups = _model_inputs_r2(dataframe)
    model = make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestClassifier(**RANDOM_FOREST_PARAMS),
    )

    features_train = features.iloc[train_idx]
    features_test = features.iloc[test_idx]
    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]
    model.fit(features_train, y_train)
    y_pred = model.predict(features_test)

    labels = sorted(y.unique())
    report = pd.DataFrame(
        classification_report(
            y_test,
            y_pred,
            labels=labels,
            output_dict=True,
            zero_division=0,
        ),
    ).transpose()
    matrix = pd.DataFrame(
        confusion_matrix(y_test, y_pred, labels=labels),
        index=pd.Index(labels, name="actual"),
        columns=pd.Index(labels, name="predicted"),
    )

    result = {
        "dataset": dataset_name,
        "split": split_name,
        "train_rows": len(train_idx),
        "test_rows": len(test_idx),
        "train_groups": groups.iloc[train_idx].nunique(),
        "test_groups": groups.iloc[test_idx].nunique(),
        "train_label_counts": dict(y_train.value_counts().sort_index()),
        "test_label_counts": dict(y_test.value_counts().sort_index()),
        "accuracy": accuracy_score(y_test, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
        "precision_macro": precision_score(
            y_test,
            y_pred,
            average="macro",
            zero_division=0,
        ),
        "recall_macro": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
        "precision_weighted": precision_score(
            y_test,
            y_pred,
            average="weighted",
            zero_division=0,
        ),
        "recall_weighted": recall_score(
            y_test,
            y_pred,
            average="weighted",
            zero_division=0,
        ),
        "f1_weighted": f1_score(y_test, y_pred, average="weighted", zero_division=0),
    }
    return result, report, matrix, model

rf_r2_results_rows = []
rf_r2_classification_reports = {}
rf_r2_confusion_matrices = {}
rf_r2_models = {}

# Iterate over each dataset and split, fit the Random Forest model, and store the results.
for dataset_name, dataframe in feature_dataframes_r2.items():
    if dataframe.empty:
        continue

    for split_name, (train_idx, test_idx) in _split_indices_r2(dataframe).items():
        model_key = (dataset_name, split_name)
        result, report, matrix, model = _fit_and_evaluate_random_forest_r2(
            dataset_name,
            split_name,
            dataframe,
            train_idx,
            test_idx,
        )
        rf_r2_results_rows.append(result)
        rf_r2_classification_reports[model_key] = report
        rf_r2_confusion_matrices[model_key] = matrix
        rf_r2_models[model_key] = model

rf_r2_results = pd.DataFrame(rf_r2_results_rows).sort_values(["dataset", "split"])
display(rf_r2_results)

for name in ["2.4 GHz", "5 GHz", "Fusion"]:
    for split in ["random", "group"]:
        print(f"\n=================== Room 2 {name} ({split} split) ===================")
        display(rf_r2_classification_reports[(name, split)])


### Position Experiments by Split Mode

Runs both group (realistic) and random (optimistic sanity-check) splits for the hierarchical and global classifiers.

In [ ]:
importlib.reload(hierarchical_position_classifier_module)

SPLIT_MODES = ("group", "random")

hier_summary, hier_preds_by_split, hier_models_by_split = run_hierarchical_position_experiments_by_split(
    feature_dataframes,
    split_modes=SPLIT_MODES,
    test_size=HIERARCHICAL_TEST_SIZE,
    random_state=HIERARCHICAL_RANDOM_STATE,
    row_spacing=HIERARCHICAL_ROW_SPACING,
    column_spacing=HIERARCHICAL_COLUMN_SPACING,
)

global_summary, global_preds_by_split, global_models_by_split = run_global_position_experiments_by_split(
    feature_dataframes,
    split_modes=SPLIT_MODES,
    test_size=HIERARCHICAL_TEST_SIZE,
    random_state=HIERARCHICAL_RANDOM_STATE,
    row_spacing=HIERARCHICAL_ROW_SPACING,
    column_spacing=HIERARCHICAL_COLUMN_SPACING,
)

display(hier_summary)
display(global_summary)


### Room-Specific Position Classifiers

Each model classifies positions inside one room only (oracle-room scenario).
 uses every ESP;  uses only ESPs assigned to that room.

**Adjust  to match the physical ESP placement in your environment.**

In [ ]:
# TODO: adjust according to the physical room layout.
# Keys are room labels (integers); values are tuples of ESP prefix strings.
ROOM_LOCAL_ESPS = {
    1: ("esp_06", "esp_07", "esp_08", "esp_09", "esp_10", "esp_16", "esp_17", "esp_18", "esp_19", "esp_20",),
    2: ("esp_01", "esp_02", "esp_03", "esp_11", "esp_12", "esp_13"),
    3: ("esp_04", "esp_05", "esp_14", "esp_15"),
}

room_summary, room_preds, room_models = run_room_specific_position_experiments(
    feature_dataframes,
    rooms=(1, 2, 3),
    esp_modes=("all", "local"),
    split_modes=SPLIT_MODES,
    room_local_esps=ROOM_LOCAL_ESPS,
    test_size=HIERARCHICAL_TEST_SIZE,
    random_state=HIERARCHICAL_RANDOM_STATE,
    row_spacing=HIERARCHICAL_ROW_SPACING,
    column_spacing=HIERARCHICAL_COLUMN_SPACING,
)

display(room_summary)


### Combined Experiment Summary

Merges global, hierarchical, and room-specific results into one comparison table.

In [ ]:
all_summary = combine_position_experiment_summaries(
    global_summary,
    hier_summary,
    room_summary,
)

display(all_summary.sort_values(["dataset", "split", "model", "room", "esp_mode"]))


### Position Error CDF by Experiment and Split

In [ ]:
for _dataset in feature_dataframes:
    for _split in SPLIT_MODES:
        all_preds_for_cdf = pd.concat(
            [
                hier_preds_by_split.get((_dataset, _split), pd.DataFrame()),
                global_preds_by_split.get((_dataset, _split), pd.DataFrame()),
            ],
            ignore_index=True,
        )
        if not all_preds_for_cdf.empty:
            plot_position_error_cdf_by_experiment(
                all_preds_for_cdf,
                dataset=_dataset,
                split=_split,
            )
